In [ ]:
import pandas as pd
import json
import ast
from google.colab import files

In [ ]:
# 2. Excel dosyasını yükle
uploaded = files.upload()


Saving npc_dataset_json_template_final_V2.xlsx to npc_dataset_json_template_final_V2.xlsx


In [ ]:
# Yüklenen dosya adını otomatik al
excel_file = list(uploaded.keys())[0]

print("Yüklenen dosya:", excel_file)

Yüklenen dosya: npc_dataset_json_template_final_V2.xlsx


In [ ]:
# 3. Excel dosyasını oku
df = pd.read_excel(excel_file, sheet_name="DATASET")


In [ ]:
# İlk satırları kontrol et
df.head()

,id,domain_id,domain,npc.role,npc.role_style,npc.roleplay_style,dialogue.question_tr,dialogue.question_en,dialogue.answer_tr,dialogue.answer_en,labels.intent,labels.player_intent,labels.emotion,labels.game_context,labels.difficulty_level,tags.tr,tags.en
0,1,EXP_PHY_MEC_001,physics,virtual_physics_teacher,explanatory,supportive,Cisimlerin hareketini değiştiren etki ne ad alır?,What is the name given to the effect that chan...,"Kuvvet, bir cismin hareketini veya şeklini değ...",Force is an effect that can change the motion ...,CONCEPT_EXPLAIN,ask_definition,informative,tutorial,very_easy,"['mekanik','temel_kavram']","['mechanic','basic_concept']"
1,2,EXP_PHY_MEC_002,physics,virtual_physics_teacher,explanatory,supportive,Kuvvet neyi belirler?,What determines the clear force?,Kuvvet cismin ivmesini belirler.,The clear force determines the acceleration of...,CONCEPT_EXPLAIN,ask_definition,informative,tutorial,very_easy,"['mekanik','kuvvet']","['mechanical','force']"
2,3,EXP_PHY_MEC_003,physics,virtual_physics_teacher,explanatory,supportive,Newton’un birinci yasası neyi açıklar?,What does Newton's first law explain?,Eylemsizlik ilkesini açıklar.,It explains the principle of inertia.,CONCEPT_EXPLAIN,ask_definition,informative,tutorial,very_easy,"['newton','hareket']","['newton','motion']"
3,4,EXP_PHY_MEC_004,physics,virtual_physics_teacher,explanatory,supportive,Newton’un ikinci yasası nedir?,What is Newton's second law?,"Kuvvet, kütle ve ivme arasındaki ilişkiyi açık...","It explains the relationship between force, ma...",CONCEPT_EXPLAIN,ask_information,informative,tutorial,easy,"['newton','ivme']","['newton','acceleration']"
4,5,EXP_PHY_MEC_005,physics,virtual_physics_teacher,explanatory,supportive,Newton’un üçüncü yasası neyi ifade eder?,What does Newton's third law state?,Etki–tepki kuvvetlerini ifade eder.,It expresses action and reaction forces.,CONCEPT_EXPLAIN,ask_information,informative,tutorial,easy,"['newton','etki_tepki']","['newton','action_response']"


In [ ]:
# 4. NaN değerleri None yap
df = df.where(pd.notnull(df), None)

In [ ]:
# 5. tags.tr ve tags.en gibi liste görünen metinleri gerçek listeye çeviren fonksiyon
def parse_possible_list(value):
    if value is None:
        return None

    if isinstance(value, list):
        return value

    if isinstance(value, str):
        value = value.strip()

        # Örnek: "['mekanik','temel_kavram']"
        if value.startswith("[") and value.endswith("]"):
            try:
                return ast.literal_eval(value)
            except Exception:
                return value

        # Alternatif: mekanik, temel_kavram
        if "," in value:
            return [item.strip() for item in value.split(",")]

    return value

In [ ]:
# 6. Noktalı sütun adlarını nested JSON yapısına çeviren fonksiyon
def set_nested_value(dictionary, dotted_key, value):
    keys = dotted_key.split(".")
    current = dictionary

    for key in keys[:-1]:
        if key not in current:
            current[key] = {}
        current = current[key]

    final_key = keys[-1]
    current[final_key] = value

In [ ]:
# 7. Her Excel satırını JSON nesnesine çevir
records = []

for _, row in df.iterrows():
    item = {}

    for column, value in row.items():
        # tags alanlarını listeye dönüştür
        if column in ["tags.tr", "tags.en"]:
            value = parse_possible_list(value)

        # id gibi sayısal alanları temizle
        if column == "id" and value is not None:
            value = int(value)

        # Noktalı alanları nested yapıya çevir
        if "." in column:
            set_nested_value(item, column, value)
        else:
            item[column] = value

    records.append(item)

print("Toplam kayıt sayısı:", len(records))
print(json.dumps(records[0], ensure_ascii=False, indent=2))

Toplam kayıt sayısı: 91720
{
  "id": 1,
  "domain_id": "EXP_PHY_MEC_001",
  "domain": "physics",
  "npc": {
    "role": "virtual_physics_teacher",
    "role_style": "explanatory",
    "roleplay_style": "supportive"
  },
  "dialogue": {
    "question_tr": "Cisimlerin hareketini değiştiren etki ne ad alır?",
    "question_en": "What is the name given to the effect that changes the motion of objects?",
    "answer_tr": "Kuvvet, bir cismin hareketini veya şeklini değiştirebilen etkidir.",
    "answer_en": "Force is an effect that can change the motion or shape of an object."
  },
  "labels": {
    "intent": "CONCEPT_EXPLAIN",
    "player_intent": "ask_definition",
    "emotion": "informative",
    "game_context": "tutorial",
    "difficulty_level": "very_easy"
  },
  "tags": {
    "tr": [
      "mekanik",
      "temel_kavram"
    ],
    "en": [
      "mechanic",
      "basic_concept"
    ]
  }
}


In [ ]:
# 8. JSONL dosyası olarak kaydet
# datetime / Timestamp gibi değerleri JSON uyumlu hale getirir

from datetime import datetime, date
import pandas as pd
import json

def json_converter(obj):
    if isinstance(obj, (datetime, date, pd.Timestamp)):
        return obj.isoformat()
    return str(obj)

output_file = "npc_dataset.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for record in records:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False,
                default=json_converter
            ) + "\n"
        )

print("JSONL dosyası oluşturuldu:", output_file)

JSONL dosyası oluşturuldu: npc_dataset.jsonl


In [ ]:
# 9. Dosyayı indir
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>